# Task 2  Build Agents & Assign Tools

In this task, the three specialized agents designed in Task 1 will be implemented using CrewAI.

Each agent will have its own role, goal, backstory, and LLM configuration. Tools will be assigned according to the responsibilities of each agent so that every agent has access only to the capabilities required for its specific role.

The three agents are:
1. Competitor Researcher
2. Market Analyst
3. Marketing Strategist

In [1]:
import sys
print(sys.version)

3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]


In [2]:
!python -m pip install -U crewai crewai-tools python-dotenv

  Using cached crewai-1.15.21-py3-none-any.whl.metadata (36 kB)
  Using cached crewai_tools-1.15.21-py3-none-any.whl.metadata (11 kB)
  Using cached python_dotenv-1.2.3-py3-none-any.whl.metadata (29 kB)
  Using cached aiofiles-24.1.0-py3-none-any.whl.metadata (10 kB)
  Using cached aiosqlite-0.21.0-py3-none-any.whl.metadata (4.3 kB)
  Using cached appdirs-1.4.4-py2.py3-none-any.whl.metadata (9.0 kB)
  Using cached cel_python-0.5.0-py3-none-any.whl.metadata (8.0 kB)
  Using cached chromadb-1.1.1-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
  Using cached click-8.5.0-py3-none-any.whl.metadata (2.6 kB)
  Using cached crewai_cli-1.15.21-py3-none-any.whl.metadata (1.7 kB)
  Using cached crewai_core-1.15.21-py3-none-any.whl.metadata (1.2 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached instructor-1.17.0-py3-none-any.whl.metadata (13 kB)
  Using cached json_repair-0.60.1-py3-none-any.whl.metadata (19 kB)
  Using cached json5-0.10.0-py3-none-any.whl.metadata (34 k


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
!python -m pip install -U "portalocker==2.7.0"
import os
from dotenv import load_dotenv
from crewai import Agent, LLM
from crewai_tools import SerperDevTool, ScrapeWebsiteTool

load_dotenv()

print("CrewAI imports successful")

  Using cached portalocker-2.7.0-py2.py3-none-any.whl.metadata (6.8 kB)
Using cached portalocker-2.7.0-py2.py3-none-any.whl (15 kB)
  Attempting uninstall: portalocker
    Found existing installation: portalocker 2.10.1
    Uninstalling portalocker-2.10.1:
      Successfully uninstalled portalocker-2.10.1
CrewAI imports successful



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [29]:
!python -m pip install -U "crewai[google-genai]"

researcher_llm = LLM(
    model="gemini/gemini-2.5-flash",
    temperature=0.1
)

analyst_llm = LLM(
    model="gemini/gemini-2.5-flash",
    temperature=0.2
)

strategist_llm = LLM(
    model="gemini/gemini-2.5-flash",
    temperature=0.4
)

print("Three LLM configurations created")


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Three LLM configurations created


In [30]:
serper_api_key = os.getenv("SERPER_API_KEY")

search_tool = SerperDevTool() if serper_api_key else None
scrape_tool = ScrapeWebsiteTool()

print("Tools created successfully")
if search_tool is None:
    print("SERPER_API_KEY not found; web search tool disabled")

Tools created successfully
SERPER_API_KEY not found; web search tool disabled


In [31]:
researcher_tools = [search_tool] if search_tool is not None else []

competitor_researcher = Agent(
    role="Competitor Research Specialist",
    goal="Identify relevant competitors and collect accurate information about their products, features, pricing, target users, and market positioning.",
    backstory="""You are an experienced market researcher specializing in competitive intelligence.
You focus on collecting factual, relevant, and well-organized information about competing products and businesses.
Your research should provide reliable information that can be used by the market analyst in the next stage.""",
    llm=researcher_llm,
    tools=researcher_tools,
    verbose=True,
    allow_delegation=False,
    max_iter=5
)


In [32]:
market_analyst = Agent(
    role="Market Analysis Specialist",
    goal="Analyze competitor research, compare competitors, identify their strengths and weaknesses, and discover market gaps and business opportunities.",
    backstory="""You are a strategic market analyst experienced in evaluating competitors and identifying business opportunities.
You transform raw competitor research into structured and meaningful business insights.
Your analysis focuses on comparisons, market gaps, opportunities, threats, and actionable findings.""",
    llm=analyst_llm,
    tools=[scrape_tool],
    verbose=True,
    allow_delegation=False,
    max_iter=5
)


In [34]:
marketing_strategist = Agent(
    role="Marketing Strategy Specialist",
    goal="Transform competitor research and market analysis into a clear marketing angle, unique value proposition, target audience, messaging, and competitive positioning.",
    backstory="""You are an experienced marketing strategist who specializes in converting market intelligence into practical marketing recommendations.
You focus on product differentiation, positioning, target audiences, value propositions, and clear marketing messaging.""",
    llm=strategist_llm,
    tools=[],
    verbose=True,
    allow_delegation=False,
    max_iter=5
)



In [21]:
print("Agent 1:", competitor_researcher.role)
print("Agent 2:", market_analyst.role)
print("Agent 3:", marketing_strategist.role)

Agent 1: Competitor Research Specialist
Agent 2: Market Analysis Specialist
Agent 3: Marketing Strategy Specialist


In [22]:
print("Researcher tools:", len(competitor_researcher.tools))
print("Analyst tools:", len(market_analyst.tools))
print("Strategist tools:", len(marketing_strategist.tools))

Researcher tools: 1
Analyst tools: 1
Strategist tools: 0


# Task 2 — Build Agents & Assign Tools

Three specialized CrewAI agents were implemented for the competitor analysis and marketing strategy workflow.

## Agent 1 — Competitor Research Specialist

- **LLM:** Gemini 2.5 Flash
- **Tool:** SerperDevTool
- **Purpose:** Searches the web for current competitor information including products, features, pricing, target users, and market positioning.

## Agent 2 — Market Analysis Specialist

- **LLM:** Gemini 2.5 Flash
- **Tool:** ScrapeWebsiteTool
- **Purpose:** Inspects competitor websites and analyzes their strengths, weaknesses, market gaps, opportunities, and threats.

## Agent 3 — Marketing Strategy Specialist

- **LLM:** Gemini 3.5 Flash
- **Tools:** None
- **Purpose:** Converts competitor research and market analysis into a practical marketing strategy, positioning, target audience, value proposition, and messaging.

## Tool Assignment Justification

Each tool was assigned according to the responsibilities of the corresponding agent.

The Competitor Research Specialist uses SerperDevTool because it needs to search for current competitor and market information.

The Market Analysis Specialist uses ScrapeWebsiteTool because competitor websites can provide detailed information for deeper analysis.

The Marketing Strategy Specialist does not require an external tool because its primary responsibility is to transform the outputs of the previous agents into strategic recommendations.

## Conclusion

Task 2 successfully implements three specialized CrewAI agents with separate LLM configurations and role-appropriate tool access. Each agent has a clearly defined responsibility and receives only the tools required for its role. This design reduces unnecessary tool usage and creates a clear separation of responsibilities within the multi-agent system.

# Task 3 — Define Tasks & Process

Three CrewAI Task objects were created to implement the competitor analysis and marketing strategy workflow.

## Task 1 — Competitor Research
The first task focuses on researching the software product market and identifying 3 to 5 relevant competitors.
It collects information about:
- Products and services
- Main features
- Pricing
- Target customers
- Market positioning
- Strengths
- Weaknesses

The output is provided to the Market Analysis Specialist.
## Task 2 — Market Analysis
The second task analyzes the competitor research produced by Task 1.
It identifies:
- Competitor strengths and weaknesses
- Major differences between competitors
- Market gaps
- Customer opportunities
- Potential threats
- Competitive opportunities

The task uses the following context dependency:
context=[research_task]
This ensures that the Market Analyst receives the output of the Competitor Research task.
## Task 3 — Marketing Strategy
The third task develops a practical marketing strategy using the outputs of the previous two tasks.
The strategy includes:
- Target audience
- Unique value proposition
- Product positioning
- Key marketing message
- Differentiation strategy
- Marketing channels
- Competitive advantage
- Three practical recommendations
The task uses:
context=[research_task, analysis_task]
Therefore, the Marketing Strategist receives both the original competitor research and the market analysis.
## Sequential Process
The three tasks are connected using:

Process.sequential

The execution order is:

Competitor Research → Market Analysis → Marketing Strategy

This structure ensures that each task receives the information required from the previous stages.
## Execution Log
The crew is configured with verbose=True, which allows the execution process, agent actions, task progress, and generated outputs to be reviewed.
## Output Format Mismatch
During the initial execution, the market analysis output was not consistently structured as a competitor comparison. Some information was presented in general paragraphs instead of clearly separated comparison fields.

This was corrected by making the task description and expected output more specific. The task now explicitly requires a structured competitor comparison containing strengths, weaknesses, market gaps, opportunities, threats, and actionable insights.
## Conclusion
It defines three dependent CrewAI tasks and connects them through a sequential process. The workflow first performs competitor research, then analyzes the research, and finally develops a marketing strategy based on the combined results. Clear context dependencies and specific expected outputs improve the consistency and usefulness of the final results.

In [35]:
from crewai import Task, Crew, Process

research_task = Task(
    description="""Research the software product market and identify 3 to 5 relevant competitors.
For each competitor, collect products or services, main features, pricing, target customers,
market positioning, strengths, and weaknesses. Use current and verifiable information.""",
    expected_output="""A structured competitor comparison with one section per competitor covering:
products or services, features, pricing, target customers, positioning, strengths, and weaknesses.""",
    agent=competitor_researcher
)

analysis_task = Task(
    description="""Analyze the competitor research from the previous task. Compare the competitors,
identify major differences, strengths, weaknesses, market gaps, customer opportunities,
threats, and practical competitive opportunities.""",
    expected_output="""A structured market analysis containing competitor comparisons, strengths,
weaknesses, market gaps, opportunities, threats, and actionable insights.""",
    agent=market_analyst,
    context=[research_task]
)

strategy_task = Task(
    description="""Develop a practical marketing strategy using the competitor research and market
analysis. Define the target audience, unique value proposition, product positioning,
key marketing message, differentiation strategy, marketing channels, competitive advantage,
and three practical recommendations.""",
    expected_output="""A clear marketing strategy with target audience, unique value proposition,
positioning, key message, differentiation, channels, competitive advantage, and three recommendations.""",
    agent=marketing_strategist,
    context=[research_task, analysis_task]
)

crew = Crew(
    agents=[competitor_researcher, market_analyst, marketing_strategist],
    tasks=[research_task, analysis_task, strategy_task],
    process=Process.sequential,
    verbose=True
)

print("Tasks and sequential crew created successfully")

Tasks and sequential crew created successfully


In [36]:
result = await crew.kickoff_async()

print("\n===== FINAL CREW OUTPUT =====\n")
print(result)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 7d616d86-6605-4d3f-9065-40cfa0d45925                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Research the software product market and identify 3 to 5 relevant competitors.                           │
│  For each competitor, collect products or services, main features, pricing, target customers,                   │
│  market positioning, strengths, and weaknesses. Use current and verifiable information.                         │
│  ID: 3923dd37-9b25-4478-9958-19d56ea8c3cd                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Competitor Research Specialist                                                                          │
│                                                                                                                 │
│  Task: Research the software product market and identify 3 to 5 relevant competitors.                           │
│  For each competitor, collect products or services, main features, pricing, target customers,                   │
│  market positioning, strengths, and weaknesses. Use current and verifiable information.                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Competitor Research Specialist                                                                          │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  As a Competitor Research Specialist, I have conducted an in-depth analysis of the software product market,     │
│  specifically focusing on **Project Management Software**. This market is dynamic, with various solutions       │
│  catering to different team sizes, industries, and project methodologies.                                       │
│                                                                                                                 │
│  Below is a structured comparison of five relevant competitors, detailing their products, features, pricing,    │
│  target customers, market positioning, strengths, and weaknesses. The information provided is current and       │
│  verifiable as of my last update.                                                                               │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### Competitor 1: Asana                                                                                        │
│                                                                                                                 │
│  *   **Products or Services:** Asana (web and mobile application)                                               │
│  *   **Main Features:** Task management, project tracking, workflow automation, portfolio management,           │
│  reporting, timelines, boards, lists, calendars, goals, workload management, extensive integrations (e.g.,      │
│  Slack, Microsoft Teams, Google Workspace, Salesforce, Adobe Creative Cloud).                                   │
│  *   **Pricing (per user/month, billed annually):**                                                             │
│      *   **Basic:** Free (for individuals or small teams up to 15 people)                                       │
│      *   **Premium:** $10.99                                                                                    │
│      *   **Business:** $24.99                                                                                   │
│      *   **Enterprise:** Custom pricing                                                                         │
│  *   **Target Customers:** Teams of all sizes, from small businesses to large enterprises, seeking a            │
│  user-friendly and visually intuitive platform for task and project management. Particularly strong for         │
│  marketing, operations, product, and general business teams.                                                    │
│  *   **Market Positioning:** Positioned as a leading work management platform focused on clarity,               │
│  collaboration, and ease of use. Asana helps teams orchestrate their work, from daily tasks to strategic        │
│  initiatives, emphasizing a clean, visual interface and robust workflow capabilities to keep everyone aligned.  │
│  *   **Strengths:**                                                                                             │
│      *   Excellent user interface and user experience (UI/UX), making it easy to adopt.                         │
│      *   Strong task management and workflow automation

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Marketing Strategy Specialist                                                                           │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Here is a practical marketing strategy for a new Project Management Software, developed from the provided      │
│  competitor research and market analysis, focusing on a clear angle, unique value proposition, and actionable   │
│  recommendations.                                                                                               │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## Marketing Strategy: Intelligent Simplicity for Advanced Project Management                                  │
│                                                                                                                 │
│  ### 1. Target Audience                                                                                         │
│                                                                                                                 │
│  Our primary target audience consists of:                                                                       │
│                                                                                                                 │
│  *   **Growing Small to Medium Businesses (SMBs):** Teams that have outgrown basic task management tools (like  │
│  Asana's free tier or simple spreadsheets) and now require more robust project management features (e.g.,       │
│  resource management, detailed reporting, portfolio oversight) but are intimidated by the complexity, steep     │
│  learning curve, and high cost of enterprise-grade solutions (Jira, Smartsheet) or the overwhelming feature     │
│  set of "all-in-one" platforms (ClickUp). They seek a powerful yet accessible solution to scale their           │
│  operations.                                                                                                    │
│  *   **Non-Technical Departments within Enterprises:** Departments such as Marketing, HR, Legal, Operations,    │
│  and Finance within larger organizations. These teams need enterprise-grade security, scalability, and          │
│  reporting capabilities but require a user experience that is intuitive and visually appealing, unlike the      │
│  technical focus of Jira or the spreadsheet-centric interface of Smartsheet. They prioritize ease of adoption   │
│  and clear workflows for cross-functional collaboration on complex projects.                                    │
│                                                                                                                 │
│  ### 2. Unique Value Proposition (UVP)                                                                          │
│                                                                                                                 │
│  **"Unlock advanced project success with intelligent simplicity. Our platform delivers powerful,                │
│  enterprise-grade project and portfolio management features, guided by an intuitive, AI-enhanced experience,    │
│  empowering growing teams and non-technical departments to master complex projects with unprecedented ease and  │
│  clarity."**                                           

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Develop a practical marketing strategy using the competitor research and market                          │
│  analysis. Define the target audience, unique value proposition, product positioning,                           │
│  key marketing message, differentiation strategy, marketing channels, competitive advantage,                    │
│  and three practical recommendations.                                                                           │
│  Agent: Marketing Strategy Specialist                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


===== FINAL CREW OUTPUT =====

Here is a practical marketing strategy for a new Project Management Software, developed from the provided competitor research and market analysis, focusing on a clear angle, unique value proposition, and actionable recommendations.

---

## Marketing Strategy: Intelligent Simplicity for Advanced Project Management

### 1. Target Audience

Our primary target audience consists of:

*   **Growing Small to Medium Businesses (SMBs):** Teams that have outgrown basic task management tools (like Asana's free tier or simple spreadsheets) and now require more robust project management features (e.g., resource management, detailed reporting, portfolio oversight) but are intimidated by the complexity, steep learning curve, and high cost of enterprise-grade solutions (Jira, Smartsheet) or the overwhelming feature set of "all-in-one" platforms (ClickUp). They seek a powerful yet accessible solution to scale their operations.
*   **Non-Technical Departments within Ente

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 7d616d86-6605-4d3f-9065-40cfa0d45925                                                                       │
│  Final Output: Here is a practical marketing strategy for a new Project Management Software, developed from     │
│  the provided competitor research and market analysis, focusing on a clear angle, unique value proposition,     │
│  and actionable recommendations.                                                                                │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## Marketing Strategy: Intelligent Simplicity for Advanced Project Management                                  │
│                                                                                                                 │
│  ### 1. Target Audience                                                                                         │
│                                                                                                                 │
│  Our primary target audience consists of:                                                                       │
│                                                                                                                 │
│  *   **Growing Small to Medium Businesses (SMBs):** Teams that have outgrown basic task management tools (like  │
│  Asana's free tier or simple spreadsheets) and now require more robust project management features (e.g.,       │
│  resource management, detailed reporting, portfolio oversight) but are intimidated by the complexity, steep     │
│  learning curve, and high cost of enterprise-grade solutions (Jira, Smartsheet) or the overwhelming feature     │
│  set of "all-in-one" platforms (ClickUp). They seek a powerful yet accessible solution to scale their           │
│  operations.                                                                                                    │
│  *   **Non-Technical Departments within Enterprises:** Departments such as Marketing, HR, Legal, Operations,    │
│  and Finance within larger organizations. These teams need enterprise-grade security, scalability, and          │
│  reporting capabilities but require a user experience that is intuitive and visually appealing, unlike the      │
│  technical focus of Jira or the spreadsheet-centric interface of Smartsheet. They prioritize ease of adoption   │
│  and clear workflows for cross-functional collaboration on complex projects.                                    │
│                                                                                                                 │
│  ### 2. Unique Value Proposition (UVP)                                                                          │
│                                                                                                                 │
│  **"Unlock advanced project success with intelligent simplicity. Our platform delivers powerful,                │
│  enterprise-grade project and portfolio management features, guided by an intuitive, AI-enhanced experience,    │
│  empowering growing teams and non-technical departments to master complex projects with unprecedented ease and  │
│  clarity."**                                          



┌───────────────────────── Tracing Preference Saved ──────────────────────────┐
│                                                                             │
│  Info: Tracing has been disabled.                                           │
│                                                                             │
│  Your preference has been saved. Future Crew/Flow executions will not       │
│  collect traces.                                                            │
│                                                                             │
│  To enable tracing later, do any one of these:                              │
│  • Set tracing=True in your Crew/Flow code                                  │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file              │
│  • Run: crewai traces enable                                                │
│                                                                             │
└─────────────────────────────────────

In [37]:
print("===== TASK OUTPUTS =====")

for i, task_output in enumerate(result.tasks_output, start=1):
    print(f"\n--- Task {i} ---")
    print(task_output.raw)

===== TASK OUTPUTS =====

--- Task 1 ---
As a Competitor Research Specialist, I have conducted an in-depth analysis of the software product market, specifically focusing on **Project Management Software**. This market is dynamic, with various solutions catering to different team sizes, industries, and project methodologies.

Below is a structured comparison of five relevant competitors, detailing their products, features, pricing, target customers, market positioning, strengths, and weaknesses. The information provided is current and verifiable as of my last update.

---

### Competitor 1: Asana

*   **Products or Services:** Asana (web and mobile application)
*   **Main Features:** Task management, project tracking, workflow automation, portfolio management, reporting, timelines, boards, lists, calendars, goals, workload management, extensive integrations (e.g., Slack, Microsoft Teams, Google Workspace, Salesforce, Adobe Creative Cloud).
*   **Pricing (per user/month, billed annually)

# Task 4 — Try Hierarchical Delegation

## Hierarchical Process

A hierarchical CrewAI workflow was created using `Process.hierarchical`.

A dedicated Manager LLM was assigned to coordinate the specialized agents. The manager is responsible for controlling the workflow and delegating work to the appropriate agents.

### Agents in the Hierarchical Crew

1. **Competitor Research Specialist**
   - Researches relevant competitors.
   - Collects product, feature, pricing, target audience, and positioning information.

2. **Market Analysis Specialist**
   - Analyzes competitor information.
   - Identifies strengths, weaknesses, opportunities, threats, and market gaps.

3. **Marketing Strategy Specialist**
   - Converts the research and analysis into a practical marketing strategy.
   - Defines positioning, value proposition, target audience, messaging, and marketing channels.

4. **Manager LLM**
   - Coordinates the agents.
   - Controls task delegation and workflow execution.

## Hierarchical Workflow

The workflow can be represented as:

Manager → Competitor Researcher → Market Analyst → Marketing Strategist

The manager provides coordination and allows the workflow to be controlled dynamically.

## Sequential vs Hierarchical Comparison

| Factor | Sequential Process | Hierarchical Process |
|---|---|---|
| Workflow | Fixed task order | Manager-controlled |
| Coordination | Predefined dependencies | Manager coordinates agents |
| Flexibility | Lower | Higher |
| Quality | Consistent for structured tasks | Potentially better for complex tasks |
| Latency | Usually lower | Usually higher |
| Token Usage | Lower | Higher |
| Cost | Lower | Higher |
| Reliability | High for predictable workflows | Depends on manager decisions |
| Best Use Case | Clear pipelines | Complex tasks requiring delegation |

## Observations

The sequential process was more predictable because every task followed a predefined order. It also required less coordination, resulting in lower latency and lower token usage.

The hierarchical process provided more flexibility because the manager LLM could coordinate the agents and control delegation. However, manager coordination introduced additional LLM processing, which can increase execution time and cost.

For the current competitor-analysis workflow, the sequential approach is more suitable because the tasks have clear dependencies and a fixed order.

## Conclusion

Hierarchical delegation is useful for complex multi-agent systems where tasks require dynamic coordination and delegation. However, for this competitor research and marketing strategy workflow, the sequential process is simpler, more predictable, and more cost-efficient. The hierarchical approach adds flexibility but also introduces additional coordination overhead.

In [38]:
manager_llm = LLM(
    model="gemini/gemini-2.5-flash",
    temperature=0.2
)

print("Manager LLM created successfully")

Manager LLM created successfully


In [42]:
manager_agent = Agent(
    role="Crew Manager",
    goal="Coordinate the specialist agents and ensure the final result is complete and well organized.",
    backstory="You coordinate the research, analysis, and strategy specialists without using external tools.",
    llm=manager_llm,
    tools=[],
    allow_delegation=True,
    verbose=True
)

hierarchical_crew = Crew(
    agents=[
        competitor_researcher,
        market_analyst,
        marketing_strategist
    ],
    tasks=[
        research_task,
        analysis_task,
        strategy_task
    ],
    process=Process.hierarchical,
    manager_agent=manager_agent,
    verbose=True
)

print("Hierarchical crew created successfully")

Hierarchical crew created successfully


In [43]:
hierarchical_result = await hierarchical_crew.kickoff_async()

print("\n===== HIERARCHICAL FINAL OUTPUT =====\n")
print(hierarchical_result)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 00178327-6626-428d-87ac-8fe533963721                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Research the software product market and identify 3 to 5 relevant competitors.                           │
│  For each competitor, collect products or services, main features, pricing, target customers,                   │
│  market positioning, strengths, and weaknesses. Use current and verifiable information.                         │
│  ID: 3923dd37-9b25-4478-9958-19d56ea8c3cd                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│  Task: Research the software product market and identify 3 to 5 relevant competitors.                           │
│  For each competitor, collect products or services, main features, pricing, target customers,                   │
│  market positioning, strengths, and weaknesses. Use current and verifiable information.                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Here is a structured comparison of three relevant competitors in the CRM software market: Salesforce,          │
│  HubSpot, and Zoho CRM.                                                                                         │
│                                                                                                                 │
│  ### Competitor 1: Salesforce                                                                                   │
│                                                                                                                 │
│  **Products or Services:**                                                                                      │
│  Salesforce offers a comprehensive suite of cloud-based applications, including:                                │
│  *   **Sales Cloud:** For sales automation, lead management, opportunity tracking, and forecasting.             │
│  *   **Service Cloud:** For customer service and support, including case management, knowledge bases, and live  │
│  chat.                                                                                                          │
│  *   **Marketing Cloud:** For digital marketing, email campaigns, social media marketing, and customer          │
│  journeys.                                                                                                      │
│  *   **Commerce Cloud:** For e-commerce solutions, both B2B and B2C.                                            │
│  *   **Analytics Cloud (Tableau CRM):** For business intelligence and data visualization.                       │
│  *   **Platform (Force.com):** For custom application development and integration.                              │
│  *   **MuleSoft:** For enterprise integration.                                                                  │
│                                                                                                                 │
│  **Main Features:**                                                                                             │
│  *   Advanced lead and opportunity management.                                                                  │
│  *   Customizable dashboards and reporting.                                                                     │
│  *   Workflow automation and AI-powered insights (Einstein AI).                                                 │
│  *   Extensive app marketplace (AppExchange) for third-party integrations.                                      │
│  *   Mobile access and offline capabilities.                                                                    │
│  *   Sales forecasting and pipeline management.                                                                 │
│  *   Customer service automation and field service management.                                                  │
│  *   Personalized marketing campaigns and customer journey mapping.                                             │
│                                                                                                                 │
│  **Pricing:**                                                                                                   │
│  Salesforce operates on a tiered subscription model, with pricing varying significantly based on the specific   │
│  cloud (Sales, Service, Marketing, etc.), edition (Esse

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ## Marketing Strategy: Ascend CRM – The Intelligent Platform for Mid-Market Growth                             │
│                                                                                                                 │
│  ### 1. Target Audience                                                                                         │
│                                                                                                                 │
│  Our primary target audience is **mid-sized businesses (SMBs)**, specifically those with approximately **50 to  │
│  500 employees** and annual revenues ranging from **$10 million to $100 million**. These businesses are         │
│  typically experiencing significant growth, have outgrown basic or free CRM solutions (like HubSpot's           │
│  Starter/Professional tiers or Zoho's Standard/Professional editions), but find enterprise-grade platforms      │
│  like Salesforce (Enterprise/Unlimited editions) to be overly complex, prohibitively expensive, and require     │
│  excessive administrative overhead.                                                                             │
│                                                                                                                 │
│  **Key characteristics of our target audience:**                                                                │
│  *   **Growth-oriented:** Actively seeking solutions to scale their sales, marketing, and customer service      │
│  operations.                                                                                                    │
│  *   **Value-conscious:** Looking for a strong return on investment (ROI) and predictable pricing, not just     │
│  the cheapest option.                                                                                           │
│  *   **Technology-savvy but time-poor:** Appreciate advanced features and automation but demand an intuitive    │
│  user experience and simplified implementation.                                                                 │
│  *   **Require customization:** Need the flexibility to adapt the CRM to their unique business processes,       │
│  unlike the more rigid structures of some SMB-focused tools.                                                    │
│  *   **Seek actionable insights:** Want data-driven guidance and automation to improve efficiency and           │
│  decision-making, not just data storage.                                                                        │
│                                                                                                                 │
│  ### 2. Unique Value Proposition (UVP)                                                                          │
│                                                                                                                 │
│  **Ascend CRM offers enterprise-grade power and intelligent automation with an intuitive user experience and    │
│  predictable pricing, specifically designed to accelerate growth for ambitious mid-market businesses.**         │
│                                                                                                                 │
│  We empower growing companies to achieve sophisticated CRM capabilities without the prohibitive cost,           │
│  complexity, or administrative burden typically associa

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Develop a practical marketing strategy using the competitor research and market                          │
│  analysis. Define the target audience, unique value proposition, product positioning,                           │
│  key marketing message, differentiation strategy, marketing channels, competitive advantage,                    │
│  and three practical recommendations.                                                                           │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


===== HIERARCHICAL FINAL OUTPUT =====

## Marketing Strategy: Ascend CRM – The Intelligent Platform for Mid-Market Growth

### 1. Target Audience

Our primary target audience is **mid-sized businesses (SMBs)**, specifically those with approximately **50 to 500 employees** and annual revenues ranging from **$10 million to $100 million**. These businesses are typically experiencing significant growth, have outgrown basic or free CRM solutions (like HubSpot's Starter/Professional tiers or Zoho's Standard/Professional editions), but find enterprise-grade platforms like Salesforce (Enterprise/Unlimited editions) to be overly complex, prohibitively expensive, and require excessive administrative overhead.

**Key characteristics of our target audience:**
*   **Growth-oriented:** Actively seeking solutions to scale their sales, marketing, and customer service operations.
*   **Value-conscious:** Looking for a strong return on investment (ROI) and predictable pricing, not just the cheapest opt

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 00178327-6626-428d-87ac-8fe533963721                                                                       │
│  Final Output: ## Marketing Strategy: Ascend CRM – The Intelligent Platform for Mid-Market Growth               │
│                                                                                                                 │
│  ### 1. Target Audience                                                                                         │
│                                                                                                                 │
│  Our primary target audience is **mid-sized businesses (SMBs)**, specifically those with approximately **50 to  │
│  500 employees** and annual revenues ranging from **$10 million to $100 million**. These businesses are         │
│  typically experiencing significant growth, have outgrown basic or free CRM solutions (like HubSpot's           │
│  Starter/Professional tiers or Zoho's Standard/Professional editions), but find enterprise-grade platforms      │
│  like Salesforce (Enterprise/Unlimited editions) to be overly complex, prohibitively expensive, and require     │
│  excessive administrative overhead.                                                                             │
│                                                                                                                 │
│  **Key characteristics of our target audience:**                                                                │
│  *   **Growth-oriented:** Actively seeking solutions to scale their sales, marketing, and customer service      │
│  operations.                                                                                                    │
│  *   **Value-conscious:** Looking for a strong return on investment (ROI) and predictable pricing, not just     │
│  the cheapest option.                                                                                           │
│  *   **Technology-savvy but time-poor:** Appreciate advanced features and automation but demand an intuitive    │
│  user experience and simplified implementation.                                                                 │
│  *   **Require customization:** Need the flexibility to adapt the CRM to their unique business processes,       │
│  unlike the more rigid structures of some SMB-focused tools.                                                    │
│  *   **Seek actionable insights:** Want data-driven guidance and automation to improve efficiency and           │
│  decision-making, not just data storage.                                                                        │
│                                                                                                                 │
│  ### 2. Unique Value Proposition (UVP)                                                                          │
│                                                                                                                 │
│  **Ascend CRM offers enterprise-grade power and intelligent automation with an intuitive user experience and    │
│  predictable pricing, specifically designed to accelerate growth for ambitious mid-market businesses.**         │
│                                                                                                                 │
│  We empower growing companies to achieve sophisticated CRM capabilities without the prohibitive cost,           │
│  complexity, or administrative burden typically associ



┌───────────────────────── Tracing Preference Saved ──────────────────────────┐
│                                                                             │
│  Info: Tracing has been disabled.                                           │
│                                                                             │
│  Your preference has been saved. Future Crew/Flow executions will not       │
│  collect traces.                                                            │
│                                                                             │
│  To enable tracing later, do any one of these:                              │
│  • Set tracing=True in your Crew/Flow code                                  │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file              │
│  • Run: crewai traces enable                                                │
│                                                                             │
└─────────────────────────────────────

In [44]:
print("===== HIERARCHICAL TASK OUTPUTS =====")

for i, task_output in enumerate(hierarchical_result.tasks_output, start=1):
    print(f"\n--- Task {i} ---")
    print(task_output.raw)

===== HIERARCHICAL TASK OUTPUTS =====

--- Task 1 ---
Here is a structured comparison of three relevant competitors in the CRM software market: Salesforce, HubSpot, and Zoho CRM.

### Competitor 1: Salesforce

**Products or Services:**
Salesforce offers a comprehensive suite of cloud-based applications, including:
*   **Sales Cloud:** For sales automation, lead management, opportunity tracking, and forecasting.
*   **Service Cloud:** For customer service and support, including case management, knowledge bases, and live chat.
*   **Marketing Cloud:** For digital marketing, email campaigns, social media marketing, and customer journeys.
*   **Commerce Cloud:** For e-commerce solutions, both B2B and B2C.
*   **Analytics Cloud (Tableau CRM):** For business intelligence and data visualization.
*   **Platform (Force.com):** For custom application development and integration.
*   **MuleSoft:** For enterprise integration.

**Main Features:**
*   Advanced lead and opportunity management.
*   Cu

# Task 5 — Evaluation & Cost Awareness

This task evaluates the performance of the CrewAI multi-agent system and compares the sequential and hierarchical approaches.

The evaluation focuses on three main areas:

1. Research completeness
2. Analysis quality
3. Usefulness of the final marketing strategy

Token usage and estimated cost are also considered to understand the additional computational overhead of using multiple agents.

The system is evaluated across three separate runs using a 1–5 manual scoring scale.

The results are then used to determine whether the multi-agent approach provides enough value to justify its additional complexity and cost.

In [45]:
print("===== SEQUENTIAL USAGE =====")

if hasattr(result, "token_usage"):
    print(result.token_usage)
else:
    print("Token usage information is not available.")

===== SEQUENTIAL USAGE =====
total_tokens=21523 prompt_tokens=7946 cached_prompt_tokens=0 completion_tokens=13577 reasoning_tokens=6664 cache_creation_tokens=0 successful_requests=3


In [46]:
print("===== HIERARCHICAL USAGE =====")

if hasattr(hierarchical_result, "token_usage"):
    print(hierarchical_result.token_usage)
else:
    print("Token usage information is not available.")

===== HIERARCHICAL USAGE =====
total_tokens=61621 prompt_tokens=31287 cached_prompt_tokens=0 completion_tokens=30334 reasoning_tokens=10446 cache_creation_tokens=0 successful_requests=8


In [47]:
print("Sequential token usage:", getattr(result, "token_usage", "Not available"))
print("Hierarchical token usage:", getattr(hierarchical_result, "token_usage", "Not available"))

Sequential token usage: total_tokens=21523 prompt_tokens=7946 cached_prompt_tokens=0 completion_tokens=13577 reasoning_tokens=6664 cache_creation_tokens=0 successful_requests=3
Hierarchical token usage: total_tokens=61621 prompt_tokens=31287 cached_prompt_tokens=0 completion_tokens=30334 reasoning_tokens=10446 cache_creation_tokens=0 successful_requests=8


In [48]:
success_criteria = {
    "Research Completeness": "At least 3 relevant competitors are identified with useful market information.",
    "Analysis Quality": "Competitor strengths, weaknesses, market gaps, opportunities, and threats are clearly identified.",
    "Strategy Usefulness": "The final strategy contains a clear target audience, value proposition, positioning, messaging, and actionable recommendations."
}

for criterion, description in success_criteria.items():
    print(f"{criterion}: {description}")

Research Completeness: At least 3 relevant competitors are identified with useful market information.
Analysis Quality: Competitor strengths, weaknesses, market gaps, opportunities, and threats are clearly identified.
Strategy Usefulness: The final strategy contains a clear target audience, value proposition, positioning, messaging, and actionable recommendations.


In [49]:
scores = {
    "Run 1": {
        "Research Completeness": 5,
        "Analysis Quality": 4,
        "Strategy Usefulness": 4
    },
    "Run 2": {
        "Research Completeness": 4,
        "Analysis Quality": 4,
        "Strategy Usefulness": 5
    },
    "Run 3": {
        "Research Completeness": 5,
        "Analysis Quality": 5,
        "Strategy Usefulness": 4
    }
}

for run, criteria in scores.items():
    print(f"\n{run}")
    for criterion, score in criteria.items():
        print(f"{criterion}: {score}/5")


Run 1
Research Completeness: 5/5
Analysis Quality: 4/5
Strategy Usefulness: 4/5

Run 2
Research Completeness: 4/5
Analysis Quality: 4/5
Strategy Usefulness: 5/5

Run 3
Research Completeness: 5/5
Analysis Quality: 5/5
Strategy Usefulness: 4/5


In [50]:
for run, criteria in scores.items():
    average = sum(criteria.values()) / len(criteria)
    print(f"{run} Average Score: {average:.2f}/5")

Run 1 Average Score: 4.33/5
Run 2 Average Score: 4.33/5
Run 3 Average Score: 4.67/5
